In [12]:
import pandas as pd
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import numpy as np

# --- 1. Dane (symulacja) ---
dates = pd.date_range(start='2024-01-01', end='2024-12-31')
np.random.seed(42)
values = np.random.randint(0, 100, size=len(dates)) # 0 pozwoli pokazać "puste" dni
df = pd.DataFrame({'date': dates, 'value': values})
df['month'] = df['date'].dt.month
df['day_of_week'] = df['date'].dt.dayofweek
df['week_grid_y'] = df['date'].dt.strftime('%W').astype(int)
df['week_grid_y'] = df.groupby('month')['week_grid_y'].transform(lambda x: x - x.min())

# --- 2. Paleta "Icy Blue Dashboard" ---
# Dopasowana do słupków na Twoim zdjęciu
def get_dashboard_color(val):
    if val == 0:
        return '#1d2026'   # Bardzo ciemny szary (puste dni - tło placeholderów)
    elif val < 30:
        return '#1e3a8a'   # Ciemny granat (mała aktywność)
    elif val < 60:
        return '#3b82f6'   # Średni niebieski
    elif val < 85:
        return '#60a5fa'   # Jasny niebieski
    else:
        return '#93c5fd'   # "Icy Blue" - kolor Twoich słupków (maksymalna aktywność)

# --- 3. Układ ---
months = [
    "Jan", "Feb", "Mar", "Apr", "May", "Jun",
    "Jul", "Aug", "Sep", "Oct", "Nov", "Dec"
]

fig = make_subplots(
    rows=2, cols=6,
    subplot_titles=months,
    horizontal_spacing=0.015,
    vertical_spacing=0.08
)

# --- 4. Rysowanie ---
for i, month_name in enumerate(months):
    month_idx = i + 1
    month_data = df[df['month'] == month_idx]
    row = 1 if month_idx <= 6 else 2
    col = month_idx if month_idx <= 6 else month_idx - 6

    # Indeksy osi
    axis_idx = (row - 1) * 6 + col
    xref = 'x' if axis_idx == 1 else f'x{axis_idx}'
    yref = 'y' if axis_idx == 1 else f'y{axis_idx}'

    hover_x, hover_y, hover_text = [], [], []

    for _, row_data in month_data.iterrows():
        w = int(row_data['week_grid_y'])
        d = int(row_data['day_of_week'])
        val = row_data['value']

        if w < 6:
            # Rysujemy "Soft Square" (Lekko zaokrąglony kwadrat)
            fig.add_shape(
                type="circle",  # Zmieniamy rect na circle
                x0=d - 0.35, x1=d + 0.35, # Nieco mniejsze niż 0.4, żeby był oddech
                y0=w - 0.35, y1=w + 0.35,
                fillcolor=get_dashboard_color(val),
                line=dict(width=0),
                xref=xref, yref=yref,
                layer="below"
            )

            # Dane do dymka
            hover_x.append(d)
            hover_y.append(w)
            hover_text.append(f"<b>{row_data['date'].strftime('%d %b')}</b><br>{val} min")

    # Niewidzialna warstwa interakcji
    fig.add_trace(
        go.Scatter(
            x=hover_x, y=hover_y,
            mode='markers',
            marker=dict(size=20, color='rgba(0,0,0,0)'),
            text=hover_text,
            hoverinfo='text',
            showlegend=False
        ),
        row=row, col=col
    )

    # Czyste osie
    fig.update_xaxes(showticklabels=False, showgrid=False, zeroline=False, range=[-0.5, 6.5], row=row, col=col)
    fig.update_yaxes(
        showticklabels=False, showgrid=False, zeroline=False,
        autorange="reversed", range=[-0.5, 5.5],
        row=row, col=col
    )

# --- 5. Styling pod Dashboard ---
fig.update_layout(
    title_text="<b>Activity Trends</b>", # Prosty, krótki tytuł
    title_x=0.01, # Tytuł wyrównany do lewej (jak "Top genre per season" na zdjęciu)
    title_font=dict(size=18, color="white", family="Arial, sans-serif"),

    # Kluczowe dla integracji z dashboardem:
    paper_bgcolor='rgba(0,0,0,0)', # Przezroczyste tło (dopasuje się do tła Twojej apki)
    plot_bgcolor='rgba(0,0,0,0)',

    font=dict(family="Arial, sans-serif", color="#a0a0a0"), # Jasnoszary tekst (subtelny)
    height=350, # Kompaktowa wysokość
    margin=dict(t=50, b=10, l=10, r=10), # Małe marginesy
    showlegend=False
)

# Dostosowanie tytułów miesięcy (subplot titles)
fig.update_annotations(font_size=12, font_color="#6b7280") # Szary, nierzucający się w oczy

fig.show()

In [26]:
import pickle

file_path = 'data/dane_ola.pkl'

# Open the file in binary mode and load the data
with open(file_path, 'rb') as file:
    data = pickle.load(file)

with open('data/dane_maciek.pkl', 'rb') as file:
    data_maciek = pickle.load(file)

In [27]:
data['conn_country'].unique()

array(['PL', 'GB', 'NL', 'IT', 'US', 'JP', 'DE', 'MT', 'ES', 'FR', 'CH',
       'AT', 'BE', 'ZZ', 'AZ', 'DK'], dtype=object)

In [28]:
data_maciek['conn_country'].unique()

array(['PL', 'ZZ', 'DE', 'AU', 'US'], dtype=object)

In [30]:
data_maciek


,platform,ms_played,conn_country,master_metadata_track_name,master_metadata_album_artist_name,master_metadata_album_album_name,spotify_track_uri,episode_name,episode_show_name,spotify_episode_uri,...,audiobook_chapter_title,reason_start,reason_end,shuffle,skipped,offline,offline_timestamp,incognito_mode,ts_date,ts_time
0,ios,165412,PL,Rah Tah Tah,"Tyler, The Creator",CHROMAKOPIA,spotify:track:5RePVWy39tLpHH0WwXgBsK,NaN,NaN,NaN,...,NaN,trackdone,trackdone,False,False,0.0,1.757331e+09,False,2025-09-08,13:34:53
1,ios,5506,PL,American Boy,Estelle,Shine,spotify:track:6etwirSInfuJrSOlogS6Sh,NaN,NaN,NaN,...,NaN,trackdone,fwdbtn,False,True,0.0,1.757331e+09,False,2025-09-08,13:35:00
2,ios,1322,PL,I Must Apologise,PinkPantheress,to hell with it,spotify:track:2rrq3SGYxk90xLx5fMmPhZ,NaN,NaN,NaN,...,NaN,fwdbtn,fwdbtn,False,True,0.0,1.757331e+09,False,2025-09-08,13:35:02
3,ios,789,PL,90210 (feat. Kacy Hill),Travis Scott,Rodeo,spotify:track:51EC3I1nQXpec4gDk0mQyP,NaN,NaN,NaN,...,NaN,fwdbtn,fwdbtn,False,True,0.0,1.757331e+09,False,2025-09-08,13:35:03
4,ios,960,PL,DTA,Casey Veggies,Sleeping in Class,spotify:track:4XLYVIMipvtkw5Ip57XeaE,NaN,NaN,NaN,...,NaN,fwdbtn,fwdbtn,False,True,0.0,1.757331e+09,False,2025-09-08,13:35:04
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
98174,ios,1258,PL,BEST INTEREST,"Tyler, The Creator",BEST INTEREST,spotify:track:3jHdKaLCkuNEkWcLVmQPCX,None,None,None,...,NaN,fwdbtn,fwdbtn,False,True,0.0,1.757331e+09,False,2025-09-08,13:24:41
98175,ios,981,PL,1-800,bbno$,1-800,spotify:track:6NBZu27n3cxEk4y4c2Kxi8,None,None,None,...,NaN,fwdbtn,fwdbtn,False,True,0.0,1.757331e+09,False,2025-09-08,13:24:42
98176,ios,1151,PL,Taking What's Not Yours,TV Girl,Who Really Cares,spotify:track:53mChDyESfwn9Dz8poHRf6,None,None,None,...,NaN,fwdbtn,fwdbtn,False,True,0.0,1.757331e+09,False,2025-09-08,13:24:43
98177,ios,157992,PL,squabble up,Kendrick Lamar,GNX,spotify:track:0nj9Bq5sHDiTxSHunhgkFb,None,None,None,...,NaN,fwdbtn,trackdone,False,False,0.0,1.757331e+09,False,2025-09-08,13:27:20


In [19]:
data.head()

,platform,ms_played,conn_country,master_metadata_track_name,master_metadata_album_artist_name,master_metadata_album_album_name,spotify_track_uri,episode_name,episode_show_name,spotify_episode_uri,...,audiobook_chapter_title,reason_start,reason_end,shuffle,skipped,offline,offline_timestamp,incognito_mode,ts_date,ts_time
0,"iOS 15.4.1 (iPad13,1)",8192,PL,Prisoner (feat. Dua Lipa),Miley Cyrus,Plastic Hearts,spotify:track:2Oycxb8QbPkpHTo8ZrmG0B,None,None,None,...,NaN,clickrow,endplay,False,False,0.0,NaN,False,2022-06-05,16:54:47
1,"iOS 15.4.1 (iPad13,1)",258969,PL,Ribs,Lorde,Pure Heroine,spotify:track:2MvvoeRt8NcOXWESkxWn3g,None,None,None,...,NaN,clickrow,trackdone,False,False,0.0,NaN,False,2022-06-05,16:59:05
2,"iOS 15.4.1 (iPad13,1)",187988,PL,Blue,Jome,Blue,spotify:track:3GaEQmo4RDdK80kszdNZN1,None,None,None,...,NaN,trackdone,trackdone,False,False,0.0,NaN,False,2022-06-05,17:02:13
3,"iOS 15.4.1 (iPad13,1)",229173,PL,Atlantis,Seafret,Tell Me It's Real,spotify:track:1Fid2jjqsHViMX6xNH70hE,None,None,None,...,NaN,trackdone,trackdone,False,False,0.0,NaN,False,2022-06-05,17:06:03
4,"iOS 15.4.1 (iPad13,1)",191106,PL,skins,The Haunting,skins,spotify:track:6Q2kn3CDlYFNwS8CAc4YzB,None,None,None,...,NaN,trackdone,trackdone,False,False,0.0,NaN,False,2022-06-05,17:09:15


In [20]:
import pandas as pd

total_ms = data['ms_played'].sum()

hours = total_ms / (1000 * 60 * 60)
days = hours / 24

print(f"Łącznie przesłuchano: {total_ms} milisekund")
print(f"To daje: {hours:.2f} godzin")
print(f"Czyli około: {days:.2f} dni ciągłego słuchania")

Łącznie przesłuchano: 66110342664 milisekund
To daje: 18363.98 godzin
Czyli około: 765.17 dni ciągłego słuchania


In [23]:
activity = data.groupby("ts_date")['ms_played'].sum()
activity.head()

ts_date
2019-05-13      296512
2019-05-14    10611024
2019-05-15    15224374
2019-05-16     8897031
2019-05-17    12702839
Name: ms_played, dtype: int64

In [22]:
import pandas as pd
kolumna_artysty = 'master_metadata_album_artist_name'

top_artists = data.groupby(kolumna_artysty)['ms_played'].sum()

top_artists_hours = (top_artists / 3600000).sort_values(ascending=False)

print(" TOP 10 wykonawcy (w godzinach):")
print(top_artists_hours.head(10))

Twoi TOP 10 wykonawcy (w godzinach):
master_metadata_album_artist_name
Gorillaz             399.276029
Melanie Martinez     352.466481
half•alive           351.137833
girl in red          288.341422
Lana Del Rey         283.262016
Taylor Swift         277.443941
Conan Gray           269.112401
Paris Paloma         261.507288
Harry Styles         248.656721
The Neighbourhood    238.785320
Name: ms_played, dtype: float64


In [13]:
import pandas as pd
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import numpy as np

def get_dashboard_color(val):
    if val == 0:
        return '#1d2026'   # Tło placeholderów (ciemny szary)
    elif val < 30:
        return '#2e1065'   # Bardzo ciemny fiolet (prawie czarny)
    elif val < 60:
        return '#7c3aed'   # Głęboki, nasycony fiolet
    elif val < 85:
        return '#c084fc'   # Jasny fiolet
    else:
        return '#e9d5ff'

months = [
    "Jan", "Feb", "Mar", "Apr", "May", "Jun",
    "Jul", "Aug", "Sep", "Oct", "Nov", "Dec"
]

fig = make_subplots(
    rows=2, cols=6,
    subplot_titles=months,
    horizontal_spacing=0.015,
    vertical_spacing=0.08
)

-
for i, month_name in enumerate(months):
    month_idx = i + 1
    month_data = df[df['month'] == month_idx]
    row = 1 if month_idx <= 6 else 2
    col = month_idx if month_idx <= 6 else month_idx - 6

    # Indeksy osi
    axis_idx = (row - 1) * 6 + col
    xref = 'x' if axis_idx == 1 else f'x{axis_idx}'
    yref = 'y' if axis_idx == 1 else f'y{axis_idx}'

    hover_x, hover_y, hover_text = [], [], []

    for _, row_data in month_data.iterrows():
        w = int(row_data['week_grid_y'])
        d = int(row_data['day_of_week'])
        val = row_data['value']

        if w < 6:
            # Rysujemy "Soft Square" (Lekko zaokrąglony kwadrat)
            fig.add_shape(
                type="circle",  # Zmieniamy rect na circle
                x0=d - 0.35, x1=d + 0.35, # Nieco mniejsze niż 0.4, żeby był oddech
                y0=w - 0.35, y1=w + 0.35,
                fillcolor=get_dashboard_color(val),
                line=dict(width=0),
                xref=xref, yref=yref,
                layer="below"
            )

            # Dane do dymka
            hover_x.append(d)
            hover_y.append(w)
            hover_text.append(f"<b>{row_data['date'].strftime('%d %b')}</b><br>{val} min")

    # Niewidzialna warstwa interakcji
    fig.add_trace(
        go.Scatter(
            x=hover_x, y=hover_y,
            mode='markers',
            marker=dict(size=20, color='rgba(0,0,0,0)'),
            text=hover_text,
            hoverinfo='text',
            showlegend=False
        ),
        row=row, col=col
    )

    # Czyste osie
    fig.update_xaxes(showticklabels=False, showgrid=False, zeroline=False, range=[-0.5, 6.5], row=row, col=col)
    fig.update_yaxes(
        showticklabels=False, showgrid=False, zeroline=False,
        autorange="reversed", range=[-0.5, 5.5],
        row=row, col=col
    )

# --- 5. Styling pod Dashboard ---
fig.update_layout(
    title_text="<b>Activity Trends</b>", # Prosty, krótki tytuł
    title_x=0.01, # Tytuł wyrównany do lewej (jak "Top genre per season" na zdjęciu)
    title_font=dict(size=18, color="white", family="Arial, sans-serif"),

    # Kluczowe dla integracji z dashboardem:
    paper_bgcolor='rgba(0,0,0,0)', # Przezroczyste tło (dopasuje się do tła Twojej apki)
    plot_bgcolor='rgba(0,0,0,0)',

    font=dict(family="Arial, sans-serif", color="#a0a0a0"), # Jasnoszary tekst (subtelny)
    height=350, # Kompaktowa wysokość
    margin=dict(t=50, b=10, l=10, r=10), # Małe marginesy
    showlegend=False
)

# Dostosowanie tytułów miesięcy (subplot titles)
fig.update_annotations(font_size=12, font_color="#6b7280") # Szary, nierzucający się w oczy

fig.show()

In [ ]:
def draw_chart(data, view_mode):
    # Kolory (Icy Blue) i Kształt (Path)
    colors = ['#1d2026', '#172554', '#1e40af', '#3b82f6', '#60a5fa', '#93c5fd']
    def get_col(v): return colors[0] if v==0 else colors[1] if v<30 else colors[2] if v<60 else colors[3] if v<120 else colors[4] if v<180 else colors[5]
    def r_rect(x,y,w,h,r): return f"M {x+r},{y} L {x+w-r},{y} Q {x+w},{y} {x+w},{y+r} L {x+w},{y+h-r} Q {x+w},{y+h} {x+w-r},{y+h} L {x+r},{y+h} Q {x},{y+h} {x},{y+h-r} L {x},{y+r} Q {x},{y} {x+r},{y} Z"

    # Logika wyświetlania (1 miesiąc vs 12 miesięcy)
    target_mo = [months.index(view_mode)+1] if view_mode != "Full Year" else range(1, 13)
    rows, cols, height = (1, 1, 500) if view_mode != "Full Year" else (2, 6, 350)
    titles = [months[i-1] for i in target_mo]

    fig = make_subplots(rows=rows, cols=cols, subplot_titles=titles, vertical_spacing=0.08, horizontal_spacing=0.015)

    for i, m in enumerate(target_mo):
        d_m = data[data['mo'] == m]
        r, c = (1, 1) if len(target_mo)==1 else (1 if i<6 else 2, i+1 if i<6 else i-5)
        xref, yref = f'x{"" if (r-1)*6+c==1 else (r-1)*6+c}', f'y{"" if (r-1)*6+c==1 else (r-1)*6+c}'

        hx, hy, ht = [], [], []
        for _, row in d_m.iterrows():
            if row['w'] < 6: # Rysujemy kafelki
                fig.add_shape(type="path", path=r_rect(row['d']-0.4, row['w']-0.4, 0.8, 0.8, 0.2),
                              fillcolor=get_col(row['mins']), line=dict(width=0), xref=xref, yref=yref, layer="below")
                hx.append(row['d']); hy.append(row['w']); ht.append(f"<b>{row['date'].strftime('%d %b')}</b><br>{int(row['mins'])} min")

        fig.add_trace(go.Scatter(x=hx, y=hy, mode='markers', marker=dict(size=20, color='rgba(0,0,0,0)'), text=ht, hoverinfo='text', showlegend=False), row=r, col=c)
        fig.update_xaxes(showticklabels=False, range=[-0.5, 6.5], row=r, col=c); fig.update_yaxes(showticklabels=False, autorange="reversed", range=[-0.5, 5.5], row=r, col=c)

    fig.update_layout(title_text=f"<b>{sel_year} Activity</b>" if view_mode=="Full Year" else f"<b>{view_mode} {sel_year}</b>",
                      paper_bgcolor='rgba(0,0,0,0)', plot_bgcolor='rgba(0,0,0,0)', font=dict(family="Arial", color="#a0a0a0"),
                      height=height, margin=dict(t=50, b=10, l=10, r=10), showlegend=False)
    fig.update_annotations(font_color="#60a5fa")
    return fig